# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nazeline-007/flyrank_internship_ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nazeline-007/flyrank_internship_ml"
REPO_DIR = "flyrank_internship_ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank_internship_ml/flyrank_internship_ml
Starter data found. You're ready.


In [13]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [14]:
print("Trend percentage summary:")
print(df["trend_pct"].describe())

print("\nTrend direction counts:")
print(df["trend_direction"].value_counts(dropna=False))


Trend percentage summary:
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [15]:
signals = [
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_pct",
    "trend_direction"
]

print(df[signals].head(10))

   search_volume   ctr  avg_position  engagement_rate  trend_pct  \
0           10.0  0.76          10.6             5.88      -41.4   
1           90.0  0.05          20.3             0.00      -57.7   
2            0.0  0.09          36.5             0.00      -60.9   
3           10.0  0.49           6.2             1.28      -13.8   
4            0.0  0.13          44.0             0.00      -34.7   
5          720.0  0.03           8.5             0.00      -38.9   
6            0.0  0.00           7.0             0.00      -92.3   
7          590.0  0.06          21.2             3.57        0.6   
8            0.0  0.09          46.0             5.88      -58.8   
9            0.0  0.16           4.9             0.00      -29.2   

  trend_direction  
0            down  
1            down  
2            down  
3          stable  
4            down  
5            down  
6            down  
7          stable  
8            down  
9            down  


## 1. My lane as an ML task (type)

**Task type: Ranking**

The goal is to rank content pages by their priority for human review.

The content team has limited time, so they cannot investigate every page. A ranking model could use multiple signals such as search demand, performance change, CTR, average position, and engagement to determine which pages should be reviewed first.

The output would be a ranked list of pages, with the highest-priority pages at the top.

In [16]:
# Check the task framing
task_type = "ranking"

print("Task type:", task_type)
assert task_type == "ranking"

print("Ranking task framing check passed.")


Task type: ranking
Ranking task framing check passed.


## 2. Target or proxy
**Target/proxy: `trend_pct`**

The target I would ideally want is a future outcome showing whether a page declines or recovers after the current observation period.

The current dataset does not provide a clean future outcome for this purpose, so I will use `trend_pct` as a **current-window proxy**. It is an observed measure of how the page's performance changed between the previous 30-day period and the latest 30-day period.

More negative values indicate a stronger decline.

This is a proxy for the current analysis and should not be interpreted as a prediction of future decline.


In [17]:
# Check the target/proxy
target = "trend_pct"

print("Target/proxy:", target)
print("\nNon-missing values:", df[target].notna().sum())
print("Missing values:", df[target].isna().sum())
print("Median trend:", df[target].median())

assert target in df.columns
assert df[target].notna().sum() > 0

print("\nTarget/proxy check passed.")


Target/proxy: trend_pct

Non-missing values: 26612
Missing values: 3388
Median trend: -33.5

Target/proxy check passed.


## 3. Success metric

**Success metric: Precision@50**

Precision@50 measures how many of the top 50 ranked pages are actually relevant to the review priority.

This metric fits the decision because the content team has limited capacity and may only be able to review a small number of pages. A good ranking should therefore place the most relevant pages near the top of the list.

A higher Precision@50 means that more of the top 50 recommendations are relevant for review.


In [18]:
# Check the success metric
success_metric = "Precision@50"

print("Success metric:", success_metric)

assert success_metric == "Precision@50"

print("Success metric check passed.")


Success metric: Precision@50
Success metric check passed.


## 4. The unit of analysis, as a real dataframe

**One row = one content/page item belonging to a client.**

Each row represents one piece of content associated with a particular client. The row contains information about that content's search demand, performance, engagement, content characteristics, and recent performance trend.

Therefore, the ranking decision is made at the individual content/page level.


In [19]:
# Show the unit of analysis using actual rows from the dataset

unit_columns = [
    "content_id",
    "client_id",
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_pct"
]

print("One row represents one content/page item:\n")
display(df[unit_columns].head(10))


One row represents one content/page item:



,content_id,client_id,search_volume,ctr,avg_position,engagement_rate,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.76,10.6,5.88,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.05,20.3,0.00,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.09,36.5,0.00,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.49,6.2,1.28,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.13,44.0,0.00,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720.0,0.03,8.5,0.00,-38.9
6,content_9a34b442b552,client_8722616204,0.0,0.00,7.0,0.00,-92.3
7,content_a63219c6e95a,client_19581e27de,590.0,0.06,21.2,3.57,0.6
8,content_5e6c160719bc,client_6208ef0f77,0.0,0.09,46.0,5.88,-58.8
9,content_c27558df2b0c,client_19581e27de,0.0,0.16,4.9,0.00,-29.2


## 5. Why ML beats a fixed rule here

A fixed rule based on a single threshold may be too rigid because review priority can depend on multiple signals together.

For example, a page with a moderate decline but very high search demand may deserve more attention than a page with a severe decline but almost no search demand. CTR, average position, and engagement rate can provide additional context.

A ranking approach can learn patterns across these signals instead of relying on one manually chosen threshold. This could help produce a more useful review order for the content team.


In [20]:
# Check the main signals used for ranking

features = [
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate"
]

print("Signals considered:")
for feature in features:
    print("-", feature)

assert all(feature in df.columns for feature in features)

print("\nML check passed.")


Signals considered:
- search_volume
- ctr
- avg_position
- engagement_rate

ML check passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.